1. 明确入库目标
   ↓
2. 读取所有 *_chunks.jsonl
   ↓
3. 每一行 JSON 转成 Document
   ↓
4. 生成稳定 ID
   ↓
5. 创建 / 重置 Chroma collection
   ↓
6. 批量 add_documents 入库
   ↓
7. 验证 collection 数量
   ↓
8. 用 similarity / MMR / metadata filter 检索验证


In [24]:
from pathlib import Path
""" 
创建一个 可以被python理解并操作的路径对象 注意r"rag_all\ingest_Chroma.ipynb" 或者 /
"""
data_dir = Path("rag_all/ingest_Chroma.ipynb")



In [25]:
# 获取当前工作目录
current_dir = Path.cwd()
print(current_dir)
print(current_dir.exists()) # Path对象可以通过exist查看是否存在

e:\2026\05\all_api\rag_all
True


In [4]:
# 这是.py文件所在的目录
file_dir = Path(__file__).parent
print(file_dir)

NameError: name '__file__' is not defined

In [27]:
""" 
地址拼接
传统方法是os.path.join
推荐做法是 / 
"""
import os
path = os.path.join(current_dir, "example.txt")
print(type(path))
print(path)
path = current_dir / "example.txt"
print(type(path))
print(path)

<class 'str'>
e:\2026\05\all_api\rag_all\example.txt
<class 'pathlib.WindowsPath'>
e:\2026\05\all_api\rag_all\example.txt


In [28]:
# 获取文件名、后缀、父目录
path = Path("rag_all/ingest_Chroma.ipynb")
print(path.name) 
print(path.suffix) 
print(path.stem)

ingest_Chroma.ipynb
.ipynb
ingest_Chroma


In [7]:
# 创建文件夹
data_dir = Path("data/raw/text")
data_dir.mkdir(parents=True, exist_ok=True)

In [29]:
# 遍历文件夹
data_dir = Path.cwd()  
for file in data_dir.iterdir():
    print(file)

e:\2026\05\all_api\rag_all\chroma
e:\2026\05\all_api\rag_all\chroma_db
e:\2026\05\all_api\rag_all\chroma_db_from_documents
e:\2026\05\all_api\rag_all\chroma_db_from_texts
e:\2026\05\all_api\rag_all\Document.png
e:\2026\05\all_api\rag_all\eval_gold_chunk.ipynb
e:\2026\05\all_api\rag_all\generated_question_bank
e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\ingest_question_bank.py
e:\2026\05\all_api\rag_all\mmr_search.png
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb
e:\2026\05\all_api\rag_all\requirements
e:\2026\05\all_api\rag_all\vectorstore.py


In [30]:
# 只会查找当前目录
for file in data_dir.glob("*.ipynb"):
    print(file)

e:\2026\05\all_api\rag_all\eval_gold_chunk.ipynb
e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb


In [10]:
# 会递归查找所有子目录
for file in data_dir.rglob("*.ipynb"):
    print(file)

e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb


![1](Document.png)

In [ ]:
from langchain_core.documents import Document
""" 
把一行Json转换成Document
这里的 obj 是一个 字典对象 dict，表示从 JSON / JSONL 文件中读取出来的一条数据。
这个函数接收一个字典 obj，再接收一个来源文件名 source_file，最后返回一个 LangChain 的 Document 对象。

复习一下 dict
dict = {
    "key1": "value1",
    "key2": "value2"}
获取数据的方法是
value1 = dict.get("key1", "default_value")
"""
def json_to_document(obj:dict , source_file: str) -> Document:
    """ 
    把一行json转换成Document格式
    """
    role = obj.get("role", "")
    topic = obj.get("topic", "")
    chunk_type = obj.get("chunk_type", "") 
    content = obj.get("content", "")
    key_points = obj.get("key_points", [])
    related_topics = obj.get("related_topics", [])
    tags = obj.get("tags", [])

    page_content = (
        f"[知识点]: {topic}\n"
        f"[知识分类]: {role}\n"
        f"[内容]: {content}\n"
        f"[关键点]: {', '.join(key_points)}\n"
        f"[相关主题]: {', '.join(related_topics)}\n"
        f"[标签]: {', '.join(tags)}\n"
    )

    metadata = {
        "source": source_file,
        "role": role,
        "topic": topic,
        "chunk_type": chunk_type,   
    }

    return Document(page_content=page_content, metadata=metadata)

In [ ]:
import json
def load_documents_and_ids_from_jsonl(file_path: Path):
    """ 
    读取单个文件地址，将所有json文件转换成Document对象
    返回一个List[Docs],ids[]
    """

    """
    打开文件地址，读取文件内容，并把每一行的 JSON 字符串转换成 Document 对象，最后返回一个 Document 对象的列表。
    处理单个jsonl文件
    返回Document 和 ids
    """
    docs = []
    ids = []
    """ 
    打开文件，读取文件内容，并在用完之后自动关闭文件 可以是Path,也可以是str
    """
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip() # 去掉字符串首尾的空白字符
            if not line:
                continue    
            obj = json.loads(line)
            """
            这行的意思是把JSON类型自动转换成合理的Python类型
            可以变成dict list等
            在这里，因为我们的json是整理好的 所以会 把一行 JSON 字符串(str)转换成 Python 字典
            load str 的意思
            是json库中的函数,用于将 JSON 格式的字符串解析成 Python 对象（通常是字典或列表）。它接受一个字符串参数，并返回解析后的 Python 对象。
            
            loads 处理的是str
            load 处理的是文件对象
            """
            doc = json_to_document(obj, source_file=file_path.name)
            docs.append(doc)
            doc_id = f"{file_path.stem}_{len(docs)}"
            ids.append(doc_id)
    return docs, ids

In [33]:
import json
from pathlib import Path
import os

from langchain_core.documents import Document

data_dir = Path.cwd()
print(type(data_dir))
data_dir = os.path.join(data_dir, "generated_question_bank")
print(type(data_dir))
data_dir = Path(data_dir)
print(type(data_dir))
print(data_dir)

""" 
generator迭代器只能被遍历一次,len(list(generator))会将generator转换成list,
之后generator就被消耗掉了,所以无法再次遍历。
所以解决方法是提前转换成list类对象
"""
# chunk_files = data_dir.glob("*_chunks.jsonl")
# print(type(chunk_files)) # 这是一个generator迭代器
# print(f"nums = {len(list(chunk_files))}")
# for chunk in chunk_files:
#     print(chunk.name)

chunk_files = list(data_dir.glob("*_chunks.jsonl"))
print(type(chunk_files)) # 这是一个generator迭代器
print(f"nums = {len(chunk_files)}")

all_docs = []
all_ids = []

for chunk in chunk_files:
    docs, ids = load_documents_and_ids_from_jsonl(chunk)
    print(f"load {chunk.name} successfully!")
    print(type(docs))
    # print(all_ids[0])
    all_docs.extend(docs)
    all_ids.extend(ids)


<class 'pathlib.WindowsPath'>
<class 'str'>
<class 'pathlib.WindowsPath'>
e:\2026\05\all_api\rag_all\generated_question_bank
<class 'list'>
nums = 9
load cpp_chunks.jsonl successfully!
<class 'list'>
load cs_fundamentals_chunks.jsonl successfully!
<class 'list'>
load embedded_chunks.jsonl successfully!
<class 'list'>
load frontend_chunks.jsonl successfully!
<class 'list'>
load go_chunks.jsonl successfully!
<class 'list'>
load java_chunks.jsonl successfully!
<class 'list'>
load llm_core_tech_chunks.jsonl successfully!
<class 'list'>
load python_backend_chunks.jsonl successfully!
<class 'list'>
load python_chunks.jsonl successfully!
<class 'list'>


In [34]:
print(f"total docs: {len(all_docs)}")
print(type(all_docs[0]))
print(all_docs[0].metadata)
print(all_ids[0])

total docs: 1374
<class 'langchain_core.documents.base.Document'>
{'source': 'cpp_chunks.jsonl', 'role': 'cpp', 'topic': 'C++ Syntax', 'chunk_type': 'principle'}
cpp_chunks_1


In [35]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

emb = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
persist_dir = "chroma"
collection_name = "knowledge_chunk"
db = Chroma(
    persist_directory=persist_dir,
    embedding_function=emb,
    collection_name=collection_name
)

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5470.60it/s]


In [36]:
db.add_documents(all_docs, ids=all_ids)

print("入库完成")
print("当前 collection 数量:", db._collection.count())

入库完成
当前 collection 数量: 1374


In [37]:
result = db._collection.get(
    ids = ["cpp_chunks_11"]
)
for doc_id , doc_text , mete in zip(result["ids"] , result["documents"],result["metadatas"]):
    print(f"ID: {doc_id}\npage_content:\n{doc_text}\nmetedata: {mete}\n")
    

ID: cpp_chunks_11
page_content:
[知识点]: C++20
[知识分类]: cpp
[内容]: C++20 引入的模块（Modules）是 C++ 语言历史上最重要的编译模型革新之一，其核心原理是彻底取代传统的头文件包含（#include）机制，通过显式声明的接口（export）和独立的编译单元来组织代码。模块在编译时被解析为独立的抽象语法树（AST），避免了头文件带来的预处理开销、宏污染、重复编译和命名空间污染等问题。模块的编译过程分为两个阶段：首先编译模块接口单元（.cppm 或 .ixx）生成模块元数据（.pcm），然后在编译其他源文件时直接导入这些元数据，无需重新解析源码。这显著提升了编译速度，尤其适用于大型项目。模块的原理依赖于 C++20 的新关键字（如 export、import、module）和编译器对模块元数据的管理，其设计目标是实现更安全的代码隔离和更快的增量构建。常见误区包括误以为模块可以完全替代头文件（实际需逐步迁移），或忽略模块接口的显式导出规则导致链接错误。工程实践中，模块适用于库开发和大型项目，但需注意工具链支持（如 MSVC、GCC 11+、Clang 10+）和构建系统（如 CMake 3.28+）的适配。
[关键点]: 模块通过 export 和 import 关键字显式定义接口，避免头文件的宏和重复编译问题。, 编译过程分为模块接口编译生成 .pcm 元数据和后续导入，提升增量构建效率。, 模块提供更好的代码隔离和安全性，减少命名冲突和预处理开销。, 工程迁移需注意工具链兼容性和构建系统配置，避免混合使用头文件和模块。, 模块不适用于所有场景，如模板特化或跨平台库可能仍需头文件辅助。
[相关主题]: 模板元编程, 编译模型与预处理, C++20 协程, 构建系统（CMake）
[标签]: C++20, 模块, 编译模型, 工程实践, 性能优化

metedata: {'role': 'cpp', 'topic': 'C++20', 'chunk_type': 'principle', 'source': 'cpp_chunks.jsonl'}



In [23]:
query = "C++ 右值引用和移动语义"
docs_found = db.similarity_search(
    query = query, 
    k=3,
)
for doc in docs_found:
    print(f"id:{doc.id}\npage_content:{doc.page_content}\nmetedata:{doc.metadata}")

id:cpp_chunks_203
page_content:[知识点]: std::forward
[知识分类]: cpp
[内容]: std::forward 是 C++11 引入的完美转发工具，其核心原理是基于引用折叠规则和模板参数推导，将函数参数的值类别（左值或右值）原样传递给其他函数，避免不必要的拷贝或移动。概念上，它用于泛型编程中，确保在模板函数内部转发参数时，保持其原始的引用类型（如左值引用或右值引用）。关键机制：当模板参数 T 被推导为左值引用（如 T&）时，std::forward<T>(arg) 返回左值引用；当 T 被推导为非引用或右值引用（如 T 或 T&&）时，返回右值引用，从而支持移动语义。常见误区包括：误用 std::forward 与 std::move 混淆，std::move 无条件转换为右值，而 std::forward 有条件保留值类别；在非模板函数中使用 std::forward 无意义，因为它依赖模板参数推导。工程实践中，std::forward 是实现完美转发模式（如工厂函数或包装器）的基础，能显著提升性能，但需注意模板参数推导的细节，避免在复杂继承场景中失效。
[关键点]: std::forward 基于引用折叠规则，保持参数的原始值类别（左值或右值）。, 模板参数 T 的推导决定转发结果：T& 为左值，T 或 T&& 为右值。, 与 std::move 不同，std::forward 是条件性的，避免无谓的移动语义。, 在模板函数中使用，确保泛型代码的完美转发，减少拷贝开销。, 常见错误：在非模板上下文使用，或忽略模板参数推导导致转发失败。
[相关主题]: std::move, 引用折叠规则, 完美转发模式, 移动语义
[标签]: C++11, 模板元编程, 内存模型, 值类别, 泛型编程

metedata:{'chunk_type': 'principle', 'source': 'cpp_chunks.jsonl', 'topic': 'std::forward', 'role': 'cpp'}
id:cpp_chunks_4
page_content:[知识点]: C++ Standards
[知识分类]: cpp
[内容]: C++ 标准（如 C++11/14/17/20/23）在工程实践中核心价值在于提升

In [38]:
query = "C++ 右值引用和移动语义"
docs_found = db.similarity_search(
    query = query, 
    k=3,
    filter={"topic":"C++11"},
)
"""
    filter={"tags":{"$contains": "C++11"}}
    这是处理同一metedata中有多个字段，但是我们只检测一个的情况
    方法是 "metadata":{"$contains":"label"}

    filter={
        "$and": [
            {"topic": ""},
            {"tags": ""}
        ]
"""
for doc in docs_found:
    print(f"id:{doc.id}\npage_content:{doc.page_content}\nmetedata:{doc.metadata}")

id:cpp_chunks_6
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的右值引用（Rvalue Reference）和移动语义（Move Semantics）是工程实践中优化资源管理的关键特性。核心概念是区分左值（有名字的持久对象）和右值（临时或即将销毁的对象），通过 `T&&` 捕获右值，实现资源的高效转移而非复制。常见应用包括自定义容器、智能指针和工厂模式中，避免不必要的深拷贝，提升性能。工程实践中，移动构造函数和移动赋值运算符通常标记为 `noexcept` 以支持标准库容器的异常安全操作。常见误区包括：1）误将左值绑定到右值引用，导致编译错误；2）忘记实现移动语义，导致类对象仍使用拷贝语义，性能低下；3）在移动后未将源对象置为有效但未指定状态，引发未定义行为；4）过度使用移动，忽略拷贝构造在某些场景下更合适（如小型对象）。面试追问角度：如何为自定义资源管理类（如文件句柄）实现移动语义？移动语义与完美转发（Perfect Forwarding）如何结合？在模板中如何避免引用折叠导致的意外行为？
[关键点]: 右值引用用于捕获临时对象，避免不必要的拷贝, 移动构造函数应转移资源所有权并置源对象为有效状态, 标记 `noexcept` 以支持标准库容器的异常安全移动, 误区：左值绑定右值引用或忽略移动语义导致性能问题, 结合完美转发在模板中高效传递参数
[相关主题]: 完美转发, 智能指针, 模板元编程, 异常安全
[标签]: 右值引用, 移动语义, C++11优化, 资源管理, 工程实践

metedata:{'chunk_type': 'practice', 'source': 'cpp_chunks.jsonl', 'topic': 'C++11', 'role': 'cpp'}
id:cpp_chunks_5
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的移动语义（Move Semantics）是解决资源管理效率问题的核心机制，其原理基于右值引用（rvalue reference）和移动构造函数/移动赋值运算符。概念上，移动语义允许将资源（如内存、文件句柄）的所有权从临时对象（右值）高效转移，而非复制，从而避免不必要的深拷贝开销。关

In [39]:
query = "C++11 引入的右值引用（Rvalue Reference）和移动语义（Move Semantics）"
docs_found = db.similarity_search_with_score(
    query = query, 
    k=3,
    # filter={"topic":"C++11"},
)
print(type(docs_found))
for doc in docs_found:
    Doc , score = doc[0] , doc[1]
    print(f"id:{Doc.id}\npage_content:{Doc.page_content}\nmetedata:{Doc.metadata}")
    print(f"score={score}")

<class 'list'>
id:cpp_chunks_196
page_content:[知识点]: Move Semantics
[知识分类]: cpp
[内容]: 移动语义（Move Semantics）是C++11引入的核心特性，旨在通过转移资源所有权而非复制来提升性能，尤其适用于临时对象或大对象场景。在工程实践中，常见应用包括实现高效容器操作（如vector扩容时移动元素）、自定义资源管理类（如智能指针、文件句柄）以及优化函数返回值（避免不必要的拷贝）。关键原理是利用右值引用（&&）和std::move将对象标记为“可移动”，从而调用移动构造函数或移动赋值运算符，转移内部资源（如指针、缓冲区）而非深拷贝。常见误区包括：误用std::move导致悬空引用（如移动后访问原对象）、在const对象上使用移动（应使用拷贝）、以及未正确实现移动操作（如遗漏noexcept修饰导致异常不安全）。工程实践建议：为资源管理类遵循Rule of Five（定义析构、拷贝、移动构造/赋值），优先使用移动语义优化性能，并通过静态分析工具检测误用。面试追问角度可聚焦于移动语义与异常安全、与完美转发的结合、以及在多线程环境下的资源转移风险。
[关键点]: 移动语义通过右值引用转移资源所有权，避免深拷贝开销。, 工程中常用于容器操作、自定义资源类和函数返回值优化。, 常见误区：误用std::move导致悬空引用或在const对象上移动。, 实践建议：遵循Rule of Five，确保移动操作noexcept以保证异常安全。, 面试可追问移动语义与完美转发、多线程资源管理的关联。
[相关主题]: 右值引用, 完美转发, Rule of Five, 异常安全
[标签]: C++11, 移动语义, 性能优化, 资源管理, 面试考点

metedata:{'role': 'cpp', 'topic': 'Move Semantics', 'source': 'cpp_chunks.jsonl', 'chunk_type': 'practice'}
score=0.6797518730163574
id:cpp_chunks_5
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的移动语义（Move Semantics）是解决资源管理效率

In [40]:
query = "C++"
docs_found = db.similarity_search_with_score(
    query = query, 
    k=3,
    # filter={"topic":"C++11"},
)
print(type(docs_found))
for doc in docs_found:
    Doc , score = doc[0] , doc[1]
    # if score < 1.0 可以做过滤
    print(f"id:{Doc.id}\npage_content:{Doc.page_content}\nmetedata:{Doc.metadata}")
    print(f"score={score}")
""" 
对比可以发现，这里的score是距离而非相似度
更精确的query的距离更短
"""

<class 'list'>
id:cpp_chunks_206
page_content:[知识点]: Range-based For
[知识分类]: cpp
[内容]: Range-based for 循环是 C++11 引入的语法糖，用于遍历容器或范围，其核心是编译器将其展开为迭代器循环。在工程实践中，它简化了代码，但需注意底层机制以避免陷阱。常见应用包括遍历 `std::vector`、`std::map` 等容器，例如 `for (auto& elem : container) { /* 使用 elem */ }`。关键原理是：循环语句 `for (auto&& elem : range)` 会被编译为 `for (auto it = std::begin(range); it != std::end(range); ++it) { auto&& elem = *it; /* 循环体 */ }`，这依赖于 `std::begin` 和 `std::end` 的 ADL（参数依赖查找）。工程实践中，必须使用引用（`auto&` 或 `auto&&`）避免不必要的拷贝，尤其对大型对象；对于 `const` 容器，使用 `const auto&`。常见误区包括：1) 遍历 `std::vector<bool>` 时，由于代理对象问题，`auto&` 可能导致编译错误，应使用 `auto` 或显式转换；2) 在循环中修改容器（如 `erase`）会导致迭代器失效，应使用 `erase-remove` 模式或 C++20 的 `std::erase_if`；3) 误用 `auto` 而非引用，导致性能下降或意外修改；4) 遍历关联容器（如 `std::map`）时，`auto` 默认是 `std::pair<const Key, Value>`，需注意 `const` 键不可修改。面试追问角度：如何自定义类型支持 range-based for？需实现 `begin()` 和 `end()` 成员函数或提供非成员版本；在多线程环境下，遍历共享容器时如何保证安全？需结合锁或原子操作。
[关键点]: Range-based for 是语法糖，编译为迭代器循环，依赖 `std::begin` 和 `std::end`。, 工程中必须使用引用（`auto&` 或 `auto

' \n对比可以发现，这里的score是距离而非相似度\n更精确的query的score更高 更长\n'

In [41]:
"""
topic更分散 
lambda_mult 0.1 更重视多样性    topic无相同
lambda_mult 0.9 更接近普通检索  实际使用中可以发现topic有相同
"""
docs_mmr = db.max_marginal_relevance_search(
    query,
    k=5,
    fetch_k=20,
    lambda_mult=0.1
)
for doc in docs_mmr:
    print(f"page_content:{doc.page_content}..., metadata={doc.metadata}, id={doc.id}")

page_content:[知识点]: Range-based For
[知识分类]: cpp
[内容]: Range-based for 循环是 C++11 引入的语法糖，用于遍历容器或范围，其核心是编译器将其展开为迭代器循环。在工程实践中，它简化了代码，但需注意底层机制以避免陷阱。常见应用包括遍历 `std::vector`、`std::map` 等容器，例如 `for (auto& elem : container) { /* 使用 elem */ }`。关键原理是：循环语句 `for (auto&& elem : range)` 会被编译为 `for (auto it = std::begin(range); it != std::end(range); ++it) { auto&& elem = *it; /* 循环体 */ }`，这依赖于 `std::begin` 和 `std::end` 的 ADL（参数依赖查找）。工程实践中，必须使用引用（`auto&` 或 `auto&&`）避免不必要的拷贝，尤其对大型对象；对于 `const` 容器，使用 `const auto&`。常见误区包括：1) 遍历 `std::vector<bool>` 时，由于代理对象问题，`auto&` 可能导致编译错误，应使用 `auto` 或显式转换；2) 在循环中修改容器（如 `erase`）会导致迭代器失效，应使用 `erase-remove` 模式或 C++20 的 `std::erase_if`；3) 误用 `auto` 而非引用，导致性能下降或意外修改；4) 遍历关联容器（如 `std::map`）时，`auto` 默认是 `std::pair<const Key, Value>`，需注意 `const` 键不可修改。面试追问角度：如何自定义类型支持 range-based for？需实现 `begin()` 和 `end()` 成员函数或提供非成员版本；在多线程环境下，遍历共享容器时如何保证安全？需结合锁或原子操作。
[关键点]: Range-based for 是语法糖，编译为迭代器循环，依赖 `std::begin` 和 `std::end`。, 工程中必须使用引用（`auto&` 或 `auto&&`）避免拷贝，提升性能。, 遍历 `std::vector<b

In [42]:
def docs_to_ids(docs: list[Document]) -> list[str]:
    retriever_ids = []
    for doc in docs:
        id = doc.id
        retriever_ids.append(id)
    return retriever_ids

In [43]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
docs = retriever.invoke("C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？")
print(type(docs))
# for doc in docs:
#     print(f"ids:{doc.id}\npage_content:{doc.page_content}\n metadata={doc.metadata}\n")
retriever_ids = docs_to_ids(docs)
print(retriever_ids)

<class 'list'>
['cpp_chunks_112', 'cpp_chunks_111', 'cpp_chunks_6', 'cpp_chunks_97']


In [44]:
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":4,
        "fetch_k":10,
        "lambda_mult":0.1,
    }
)

docs = retriever.invoke(query)
for doc in docs:
    print(doc.page_content, doc.metadata, doc.id)

[知识点]: Range-based For
[知识分类]: cpp
[内容]: Range-based for 循环是 C++11 引入的语法糖，用于遍历容器或范围，其核心是编译器将其展开为迭代器循环。在工程实践中，它简化了代码，但需注意底层机制以避免陷阱。常见应用包括遍历 `std::vector`、`std::map` 等容器，例如 `for (auto& elem : container) { /* 使用 elem */ }`。关键原理是：循环语句 `for (auto&& elem : range)` 会被编译为 `for (auto it = std::begin(range); it != std::end(range); ++it) { auto&& elem = *it; /* 循环体 */ }`，这依赖于 `std::begin` 和 `std::end` 的 ADL（参数依赖查找）。工程实践中，必须使用引用（`auto&` 或 `auto&&`）避免不必要的拷贝，尤其对大型对象；对于 `const` 容器，使用 `const auto&`。常见误区包括：1) 遍历 `std::vector<bool>` 时，由于代理对象问题，`auto&` 可能导致编译错误，应使用 `auto` 或显式转换；2) 在循环中修改容器（如 `erase`）会导致迭代器失效，应使用 `erase-remove` 模式或 C++20 的 `std::erase_if`；3) 误用 `auto` 而非引用，导致性能下降或意外修改；4) 遍历关联容器（如 `std::map`）时，`auto` 默认是 `std::pair<const Key, Value>`，需注意 `const` 键不可修改。面试追问角度：如何自定义类型支持 range-based for？需实现 `begin()` 和 `end()` 成员函数或提供非成员版本；在多线程环境下，遍历共享容器时如何保证安全？需结合锁或原子操作。
[关键点]: Range-based for 是语法糖，编译为迭代器循环，依赖 `std::begin` 和 `std::end`。, 工程中必须使用引用（`auto&` 或 `auto&&`）避免拷贝，提升性能。, 遍历 `std::vector<bool>` 时，因代理对象

# RAG效果评估

## 指标
### 1。Recall 
前k个检索结果中，是否包含正确的文档

### 2.Precision@k
前k个结果中有多少是真的相关的

### 3.MRR
正确答案排在第几位

### 4，nDCG
强相关 - 中等相关 
相关结果是否排列靠后

### 5.Hit rate 
前k个结果中是否命中至少一个相关文档

query -> gold chunk 就是一个测试问题，对应哪些chunk应该被检索出来
query - gold chunk的作用是

In [45]:
# 一个 query 对多个 gold chunks。
# gold 字典的 key 是 chunk_id，value 是人工标注的相关性等级。
# 3 = 核心答案，2 = 强相关补充，1 = 弱相关背景。

eval_set = [
    {
        "query_id": "eval_cpp_001",
        "query": "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？",
        "expected_role": "cpp",
        "expected_topic": ["Use-After-Free", "RAII", "Smart Pointers", "AddressSanitizer", "Valgrind"],
        "gold": {
            "cpp_chunks_112": 3,
            "cpp_chunks_111": 3,
            "cpp_chunks_88": 2,
            "cpp_chunks_90": 2,
            "cpp_chunks_170": 1,
            "cpp_chunks_256": 1,
            "cpp_chunks_254": 1,
        },
    },
    {
        "query_id": "eval_cs_001",
        "query": "设计缓存或数据库索引时，哈希表如何实现平均 O(1) 查询？冲突、负载因子和并发安全应该如何处理？",
        "expected_role": "cs_fundamentals",
        "expected_topic": ["Hash Tables", "B-Trees"],
        "gold": {
            "cs_fundamentals_chunks_22": 3,
            "cs_fundamentals_chunks_21": 3,
            "cs_fundamentals_chunks_18": 1,
            "cs_fundamentals_chunks_17": 1,
        },
    },
    {
        "query_id": "eval_embedded_001",
        "query": "在资源受限的嵌入式设备上部署边缘 AI 模型时，如何在精度、延迟、内存和功耗之间做权衡？",
        "expected_role": "embedded",
        "expected_topic": ["Model Deployment", "Edge AI", "Quantization", "Model Compression"],
        "gold": {
            "embedded_chunks_38": 3,
            "embedded_chunks_37": 3,
            "embedded_chunks_2": 2,
            "embedded_chunks_1": 2,
            "embedded_chunks_6": 2,
            "embedded_chunks_5": 2,
            "embedded_chunks_8": 1,
            "embedded_chunks_7": 1,
        },
    },
    {
        "query_id": "eval_frontend_001",
        "query": "一个前端页面首屏加载慢、交互卡顿时，应该从资源加载、渲染、缓存和主线程执行哪些方面做性能优化？",
        "expected_role": "frontend",
        "expected_topic": ["Performance Optimization", "Browser Rendering", "Event Loop", "DOM Manipulation"],
        "gold": {
            "frontend_chunks_20": 3,
            "frontend_chunks_19": 3,
            "frontend_chunks_22": 2,
            "frontend_chunks_21": 2,
            "frontend_chunks_24": 2,
            "frontend_chunks_23": 1,
            "frontend_chunks_36": 1,
        },
    },
    {
        "query_id": "eval_go_001",
        "query": "Go 服务中如何用 context 管理 HTTP 请求、数据库调用和 goroutine 的超时取消，避免资源泄漏？",
        "expected_role": "go",
        "expected_topic": ["Context", "Goroutines", "Channels", "Select"],
        "gold": {
            "go_chunks_122": 3,
            "go_chunks_121": 3,
            "go_chunks_70": 2,
            "go_chunks_69": 2,
            "go_chunks_104": 1,
            "go_chunks_103": 1,
            "go_chunks_110": 1,
        },
    },
    {
        "query_id": "eval_java_001",
        "query": "Java 高并发服务频繁 Full GC 和 STW 停顿时，应该如何监控、分析并选择合适的 GC 调优策略？",
        "expected_role": "java",
        "expected_topic": ["Garbage Collection", "JVM", "Performance Tuning", "Concurrency"],
        "gold": {
            "java_chunks_8": 3,
            "java_chunks_7": 3,
            "java_chunks_6": 2,
            "java_chunks_5": 2,
            "java_chunks_50": 2,
            "java_chunks_49": 2,
            "java_chunks_12": 1,
        },
    },
    {
        "query_id": "eval_llm_001",
        "query": "构建 RAG 系统时，向量数据库应该如何负责 embedding 存储、相似度检索、索引构建和结果后处理？",
        "expected_role": "llm_core_tech",
        "expected_topic": [
            "Retrieval-Augmented Generation (RAG)",
            "Vector Databases",
            "Embeddings Retrieval",
            "Context Window Optimization",
        ],
        "gold": {
            "llm_core_tech_chunks_44": 3,
            "llm_core_tech_chunks_43": 3,
            "llm_core_tech_chunks_42": 3,
            "llm_core_tech_chunks_41": 2,
            "llm_core_tech_chunks_46": 2,
            "llm_core_tech_chunks_45": 2,
            "llm_core_tech_chunks_48": 1,
        },
    },
    {
        "query_id": "eval_py_backend_001",
        "query": "FastAPI 项目中如何利用类型提示、Pydantic、依赖注入、ASGI 和 async/await 构建高性能 REST API？",
        "expected_role": "python_backend",
        "expected_topic": ["FastAPI", "ASGI", "Uvicorn", "Starlette", "REST APIs"],
        "gold": {
            "python_backend_chunks_2": 3,
            "python_backend_chunks_1": 3,
            "python_backend_chunks_14": 2,
            "python_backend_chunks_13": 2,
            "python_backend_chunks_12": 2,
            "python_backend_chunks_11": 1,
            "python_backend_chunks_18": 1,
            "python_backend_chunks_17": 1,
        },
    },
    {
        "query_id": "eval_python_001",
        "query": "CPython 的 GIL 为什么会影响 CPU 密集型多线程性能？实际项目中应如何通过多进程、异步或线程池方案规避？",
        "expected_role": "python",
        "expected_topic": ["GIL", "Multiprocessing", "Async IO", "asyncio", "Multithreading", "threading"],
        "gold": {
            "python_chunks_94": 3,
            "python_chunks_93": 3,
            "python_chunks_128": 2,
            "python_chunks_127": 2,
            "python_chunks_116": 2,
            "python_chunks_114": 2,
            "python_chunks_132": 1,
            "python_chunks_134": 1,
        },
    },
]
print(type(eval_set))
print(type(eval_set[0]))
len(eval_set)

<class 'list'>
<class 'dict'>


9

In [46]:
import math


def get_relevant_ids(gold, relevant_threshold):
    """从gold标准答案中取出真正相关的文档。"""
    relevant_ids = set()
    for doc_id , rel in gold.items():
        if rel >= relevant_threshold:
            relevant_ids.add(doc_id)

    return relevant_ids
    # return {doc_id for doc_id, rel in gold.items() if rel >= relevant_threshold}


def precision_at_k(retrieved_ids, gold, k=5, relevant_threshold=1):
    """
    Precision@k = top-k 里命中 gold 的数量 / k。
    除以的是检索topk的数量
    """
    topk = retrieved_ids[:k]
    relevant_ids = get_relevant_ids(gold, relevant_threshold) # 取出真正相关的文档ID
    hit_count = sum(1 for doc_id in topk if doc_id in relevant_ids)
    return hit_count / k


def recall_at_k(retrieved_ids, gold, k=5, relevant_threshold=1):
    """
    Recall@k = top-k 里命中的 gold 数量 / 所有应命中的 gold 数量。
    除以的是相关chunk的数量
    """
    topk = retrieved_ids[:k]
    relevant_ids = get_relevant_ids(gold, relevant_threshold)
    if len(relevant_ids) == 0:
        return 0
    hit_count = sum(1 for doc_id in topk if doc_id in relevant_ids)
    return hit_count / len(relevant_ids)


def mrr_at_k(retrieved_ids, gold, k=5, relevant_threshold=2):
    """MRR@k = 第一个命中结果排名的倒数。第一名命中就是 1，第二名命中就是 1/2。"""
    relevant_ids = get_relevant_ids(gold, relevant_threshold)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0


def dcg_at_k(retrieved_ids, gold:dict, k=5):
    """DCG@k = 按排名折损后的相关性得分。越相关、越靠前，贡献越大。"""
    score = 0
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        rel = gold.get(doc_id, 0)
        score += rel / math.log2(rank + 1)
    return score


def ndcg_at_k(retrieved_ids, gold, k=5):
    """nDCG@k = 当前排序的 DCG / 理想排序的 DCG。范围通常是 0 到 1。"""
    dcg = dcg_at_k(retrieved_ids, gold, k)
    ideal_rels = sorted(gold.values(), reverse=True)
    ideal_ids = [f"ideal_{i}" for i in range(len(ideal_rels))]
    ideal_gold = dict(zip(ideal_ids, ideal_rels))
    ideal_dcg = dcg_at_k(ideal_ids, ideal_gold, k)
    if ideal_dcg == 0:
        return 0
    return dcg / ideal_dcg


def evaluate_one(retrieved_ids, gold, k=5, relevant_threshold=2):
    """对单个 query 的检索结果计算四个指标。"""
    return {
        f"precision@{k}": precision_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"recall@{k}": recall_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"mrr@{k}": mrr_at_k(retrieved_ids, gold, k, relevant_threshold),
        f"ndcg@{k}": ndcg_at_k(retrieved_ids, gold, k),
    }


def average_metrics(rows):
    """对多条 query 的指标求平均。"""
    if not rows:
        return {}
    metric_names = rows[0].keys()
    avg = {}
    for name in metric_names:
        avg[name] = sum(row[name] for row in rows) / len(rows)
    return avg


In [47]:
score = evaluate_one(retriever_ids,gold=eval_set[0]["gold"],k=5,relevant_threshold=1)
print(score)

{'precision@5': 0.4, 'recall@5': 0.2857142857142857, 'mrr@5': 1.0, 'ndcg@5': 0.6851691024258786}


In [15]:
def route_query(query:str):
    """ 
    根据用户问题 判断应该检索哪个roll
    方法是 关键词 关键词存在于query 则认为query属于该类问题
    返回值是一个dict {role:result} 可以用于filter
    """
    q = query.lower()
    role_keywords = {
        "cpp": ["c++", "cpp", "右值", "raii", "智能指针", "析构", "std::", "use-after-free"],
        "python": ["cpython", "gil", "python 语法", "asyncio", "multiprocessing", "threading"],
        "python_backend": ["fastapi", "django", "flask", "asgi", "wsgi", "uvicorn", "rest api"],
        "java": ["java", "jvm", "gc", "full gc", "spring", "并发编程"],
        "go": ["go ", "golang", "goroutine", "channel", "context", "gmp"],
        "frontend": ["前端", "浏览器", "react", "vue", "javascript", "event loop", "渲染"],
        "embedded": ["嵌入式", "边缘", "edge ai", "量化", "onnx", "mcu", "soc"],
        "llm_core_tech": ["rag", "embedding", "向量数据库", "transformer", "attention", "llm"],
        "cs_fundamentals": ["哈希表", "数组", "链表", "图", "b-tree", "数据结构", "算法"],
    }
    for role , keywords in role_keywords.items():
        for kw in keywords:
            if kw in q:
                return {"role":role}
            
    return None

In [22]:
query1 = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
route_filter = route_query(query)
query2 = eval_set[2].get("query")
route_filter2 = route_query(query2)
print(route_filter2)

{'role': 'embedded'}


In [23]:
def routed_search(query,k=5):
    route_filter = route_query(query)
    docs = db.similarity_search(query,k=k,filter=route_filter)
    return docs


In [48]:
role_keywords = {
    "cpp": ["c++", "cpp", "右值", "raii", "智能指针", "析构", "std::", "use-after-free"],
    "python": ["cpython", "gil", "python 语法", "asyncio", "multiprocessing", "threading"],
    "python_backend": ["fastapi", "django", "flask", "asgi", "wsgi", "uvicorn", "rest api"],
    "java": ["java", "jvm", "gc", "full gc", "spring", "并发编程"],
    "go": ["go ", "golang", "goroutine", "channel", "context", "gmp"],
    "frontend": ["前端", "浏览器", "react", "vue", "javascript", "event loop", "渲染"],
    "embedded": ["嵌入式", "边缘", "edge ai", "量化", "onnx", "mcu", "soc"],
    "llm_core_tech": ["rag", "embedding", "向量数据库", "transformer", "attention", "llm"],
    "cs_fundamentals": ["哈希表", "数组", "链表", "图", "b-tree", "数据结构", "算法"],
}

""" 
原来是这样的dict
role_keywords = {
    "cpp": ["c++", "cpp", "右值", "raii"],
    "python": ["cpython", "gil", "python 语法"],
}
dict_items = role_keywords.items() 会得到一个特殊的dict_item类型的对象，这个对象可以被迭代，但是不能直接取元素
如果要取出某元素需要转换成list 这也是常规操作
dict_items([
    ("cpp", ["c++", "cpp", "右值", "raii"]),
    ("python", ["cpython", "gil", "python 语法"])
])
里面的每个元素都是tuple 类型，所以需要先转换成list 再取元素
"""
print(type(role_keywords))
dict_items = list(role_keywords.items())
print(type(dict_items))
item = dict_items[0]
print(type(item))
print(item)
print(item[0])
print(item[1])


<class 'dict'>
<class 'list'>
<class 'tuple'>
('cpp', ['c++', 'cpp', '右值', 'raii', '智能指针', '析构', 'std::', 'use-after-free'])
cpp
['c++', 'cpp', '右值', 'raii', '智能指针', '析构', 'std::', 'use-after-free']


In [14]:
role_keywords = {
    "cpp": ["c++", "cpp", "右值", "raii", "智能指针", "析构", "std::", "use-after-free"],
    "python": ["cpython", "gil", "python 语法", "asyncio", "multiprocessing", "threading"],
    "python_backend": ["fastapi", "django", "flask", "asgi", "wsgi", "uvicorn", "rest api"],
    "java": ["java", "jvm", "gc", "full gc", "spring", "并发编程"],
    "go": ["go ", "golang", "goroutine", "channel", "context", "gmp"],
    "frontend": ["前端", "浏览器", "react", "vue", "javascript", "event loop", "渲染"],
    "embedded": ["嵌入式", "边缘", "edge ai", "量化", "onnx", "mcu", "soc"],
    "llm_core_tech": ["rag", "embedding", "向量数据库", "transformer", "attention", "llm"],
    "cs_fundamentals": ["哈希表", "数组", "链表", "图", "b-tree", "数据结构", "算法"],
}
print(type(role_keywords.items()))
for role , keywords in role_keywords.items():
    print(type(role))

<class 'dict_items'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>


###  query 的结果
全流程是：
首先，查询路由表，获取到相关的文档集合

然后，使用检索器从这些文档集合中提取出相关的文档   --- routed_search

这里得到的是文档，转换成docs_id                  --- docs_to_ids

最后，通过evaluate函数评估


In [59]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
docs = db.similarity_search(query=query,k=5)
retriever_ids = docs_to_ids(docs)
print(retriever_ids)
score = evaluate_one(retriever_ids,gold=eval_set[0]["gold"],k=5,relevant_threshold=1)
print(score)

['cpp_chunks_112', 'cpp_chunks_111', 'cpp_chunks_6', 'cpp_chunks_97', 'cpp_chunks_26']
{'precision@5': 0.4, 'recall@5': 0.2857142857142857, 'mrr@5': 1.0, 'ndcg@5': 0.6851691024258786}


In [84]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
docs = routed_search(query=query,k=5)
retriever_ids = docs_to_ids(docs)
print(type(retriever_ids))
print(retriever_ids)
score = evaluate_one(retriever_ids,gold=eval_set[0]["gold"],k=5,relevant_threshold=1)
print(score)

<class 'list'>
['cpp_chunks_112', 'cpp_chunks_111', 'cpp_chunks_6', 'cpp_chunks_97', 'cpp_chunks_26']
{'precision@5': 0.4, 'recall@5': 0.2857142857142857, 'mrr@5': 1.0, 'ndcg@5': 0.6851691024258786}


In [55]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],  # 或者用你已读取出来的 DeepSeek_API_KEY 变量
    base_url="https://api.deepseek.com",
    temperature=0,
)
response = llm.invoke([
    HumanMessage(content="你是什么模型")
])
print(response)
print(response.content)
print(type(response))

content='你好！我是 **DeepSeek**，由深度求索公司开发的AI助手。我是最新版本的模型，知识截止时间为2025年5月。\n\n我的一些特点：\n- **纯文本模型**，但支持阅读链接和上传文件（图片、PDF、Word、Excel、PPT等）并从中提取文字信息\n- **超长上下文**（1M tokens），可以一次性处理像《三体》三部曲那样的大部头书籍\n- **免费使用**，支持联网搜索（需手动开启）\n- 提供Web端和App端服务，App端还支持语音输入\n\n有什么我可以帮你的吗？无论是回答问题、写作、编程还是其他需求，尽管问我！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 301, 'prompt_tokens': 7, 'total_tokens': 308, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 153, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a084a68f-8e6e-4728-8b45-b62708426368', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1a44-dc69-7b12-8809-d8a500cf1579-0' tool_calls=[] invalid_to

In [61]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

multi_query_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
你是一个 RAG 检索 query 改写助手。

你的任务：
把用户的原始问题改写成多个适合向量检索的 query。

要求：
1. 不要回答问题，只生成检索 query。
2. 保留原问题中的关键技术词，例如 C++、GIL、FastAPI、RAG、ASGI、std::move。
3. 生成的 query 要覆盖不同检索角度：
   - 原理机制
   - 工程实践
   - 常见问题/误区
   - 性能优化/排障
4. query 不要太长，每条控制在 15 到 40 个中文词之间。
5. 输出必须是 JSON 数组，不要 Markdown，不要解释。

输出示例：
[
  "C++ Use-After-Free 触发场景 RAII 智能指针",
  "C++ 释放后使用问题 如何避免 悬空指针",
  "C++ 多线程场景 Use-After-Free 同步机制",
  "C++ AddressSanitizer Valgrind 检测 UAF"
]
"""
    ),
    ("user", "原始问题：{query}")

])
multi_query_chain = multi_query_prompt | llm | StrOutputParser()

In [66]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
query_text = multi_query_chain.invoke(query)
print(type(query_text))
print(query_text)

<class 'langchain_core.messages.base.TextAccessor'>
[
  "C++ Use-After-Free 触发场景 内存释放后 访问 悬空指针",
  "C++ Use-After-Free 原理 未定义行为 内存管理 漏洞",
  "C++ RAII 智能指针 避免 Use-After-Free 工程实践",
  "C++ 多线程 Use-After-Free 同步机制 条件竞争 避免方法",
  "C++ 检测 Use-After-Free AddressSanitizer Valgrind 工具"
]


In [ ]:
import re
def parse_query_list(text:str):
    """ 
    把llm的输出解析成为list[str]
    不一定用的上 也不一定有用
    """
    text = text.strip()
    text = text.replace("json","").replace("'''","").strip()
    """ 
    re.search(pattern, text) 在text中寻找第一个匹配pattern的内容
    match.group(0) 表示：取出正则匹配到的完整文本。
    """
    match = re.search(r"\[.*\]", text, flags=re.S)
    if match:
        text = match.group(0)
    try:
        data = json.loads(text)
        if isinstance(data,list): # 判断data是不是一个列表
            return [str(x).strip() for x in data if str(x).strip()]
    except Exception:
        pass

    return None

In [70]:
generated_queries = parse_query_list(query_text)
print(generated_queries)

['C++ Use-After-Free 触发场景 内存释放后 访问 悬空指针', 'C++ Use-After-Free 原理 未定义行为 内存管理 漏洞', 'C++ RAII 智能指针 避免 Use-After-Free 工程实践', 'C++ 多线程 Use-After-Free 同步机制 条件竞争 避免方法', 'C++ 检测 Use-After-Free AddressSanitizer Valgrind 工具']


In [71]:
def generate_multi_query(query:str , max_querys =5):
    """ 
    生成multi-query，合并在一起
    """
    raw_text = multi_query_chain.invoke(query)
    generated_queries = parse_query_list(raw_text)
    queries = [query] + generated_queries

    unique_queries = []
    for query in queries:
        q = query.strip()
        if q and q not in unique_queries:
            unique_queries.append(q)
    return unique_queries

In [74]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
generated_queries = generate_multi_query(query,5)
for q in generated_queries:
    print(q)

C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？
C++ Use-After-Free 触发场景 悬挂指针 野指针
RAII 智能指针 避免 C++ 释放后使用 工程实践
C++ 多线程 Use-After-Free 同步机制 互斥锁
C++ Use-After-Free 性能优化 内存管理 安全策略


In [ ]:
def generate_multi_query_and_search(query,k_per_query=5,final_k=5,rrf_k=60):
    """ 
    Multi-Query + rag-fusion
    调用generate_multi_query生成多个query
    每个query分别检索
    合并去重 取出前5

    rrf评分规则是score += 1 / (rrf_k + rank)
    """
    # route_filter = route_query(query)
    doc_map = {}
    rrf_scores = {}
    generated_queries = generate_multi_query(query,5)
    for query in generated_queries:
        docs_found_per_query = db.similarity_search(query,5)
        for rank,doc in enumerate(docs_found_per_query,start=1): # start=1的意思是排名从1开始 而不是默认的从0开始
            doc_map[doc.id] = doc
            rrf_scores[doc.id] = rrf_scores.get(doc.id,0) + 1 / (rrf_k + rank)
            print(f"id:{doc.id},")

    sorted_doc_ids = sorted(rrf_scores,key=lambda doc_id:rrf_scores[doc_id],reverse=True)
    # 默认是按照dict的key排序
    # reverse = True 表示按从大到小排序
    doc_id = sorted_doc_ids[:final_k]
    return doc_id

In [83]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
ids = generate_multi_query_and_search(query)
print(ids)

id:cpp_chunks_112,
id:cpp_chunks_111,
id:cpp_chunks_6,
id:cpp_chunks_97,
id:cpp_chunks_26,
id:cpp_chunks_112,
id:cpp_chunks_111,
id:cpp_chunks_97,
id:cpp_chunks_239,
id:cpp_chunks_28,
id:cpp_chunks_6,
id:cpp_chunks_112,
id:cpp_chunks_28,
id:cpp_chunks_62,
id:cpp_chunks_89,
id:cpp_chunks_112,
id:cpp_chunks_6,
id:cpp_chunks_168,
id:cpp_chunks_169,
id:cpp_chunks_10,
id:cpp_chunks_112,
id:cpp_chunks_111,
id:cpp_chunks_88,
id:cpp_chunks_48,
id:cpp_chunks_118,
id:cpp_chunks_89,
id:cpp_chunks_95,
id:cpp_chunks_94,
id:cpp_chunks_96,
id:cpp_chunks_90,
['cpp_chunks_112', 'cpp_chunks_6', 'cpp_chunks_111', 'cpp_chunks_89', 'cpp_chunks_97']


In [85]:
print(type(ids))

<class 'list'>


In [86]:
score = evaluate_one(ids,gold=eval_set[0]["gold"],k=5,relevant_threshold=1)
print(score)

{'precision@5': 0.4, 'recall@5': 0.2857142857142857, 'mrr@5': 1.0, 'ndcg@5': 0.6301642675830844}


In [88]:
from sentence_transformers import CrossEncoder

# 第一次会加载模型。可以换成你本地已有的 reranker。
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",device="cuda")

e:\miniconda\envs\langchain2\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yifei\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7485.50it/s]


In [89]:
def generate_multi_query_and_rerank(query,k_per_query=5,final_k=5):
    doc_map={}
    pairs = []
    
    generated_queries = generate_multi_query(query,5)
    for q in generated_queries:
        docs_found_per_query = db.similarity_search(q,k=k_per_query)
        for doc in docs_found_per_query:
            doc_map[doc.id]=doc
    
    docs = list(doc_map.values())

    for doc in docs:
        pairs.append([query,doc.page_content])

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))

    scored_docs = sorted(
        scored_docs,
        key=lambda x: x[1],
        reverse=True
    )

    return scored_docs[:final_k]
    
    

In [92]:
query = "C++ 工程中出现 Use-After-Free 问题时，通常有哪些触发场景，应该如何用 RAII、智能指针和同步机制避免？"
results = generate_multi_query_and_rerank(query)
ids = []
for doc, score in results:
    ids.append(doc.id)
    print("id:", doc.id)
    print("score:", score)
    print("-" * 50)
print(ids)

id: cpp_chunks_111
score: 8.356651
--------------------------------------------------
id: cpp_chunks_112
score: 8.065802
--------------------------------------------------
id: cpp_chunks_97
score: 5.215418
--------------------------------------------------
id: cpp_chunks_6
score: 5.155343
--------------------------------------------------
id: cpp_chunks_3
score: 5.0790076
--------------------------------------------------
['cpp_chunks_111', 'cpp_chunks_112', 'cpp_chunks_97', 'cpp_chunks_6', 'cpp_chunks_3']


In [93]:
score = evaluate_one(ids,gold=eval_set[0]["gold"],k=5,relevant_threshold=1)
print(score)

{'precision@5': 0.4, 'recall@5': 0.2857142857142857, 'mrr@5': 1.0, 'ndcg@5': 0.6851691024258786}
